In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.colors

In [3]:
pudl = pd.read_csv("Mapping/pudl_with_profile_names.csv")

## Assign units to ungrouped LRZs
conditions = [
    (pudl["BA"] == "MISO-0001"),
    (pudl["BA"] == "MISO-0027") & (pudl["state"] == "WI"),
    (pudl["BA"] == "MISO-0027") & (pudl["state"] == "MI"),
    (pudl["BA"] == "MISO-0035") & pudl["state"].isin(["IA", "MN", "IL"]),
    (pudl["BA"] == "MISO-0035"),
    (pudl["BA"] == "MISO-0004"),
    (pudl["BA"] == "MISO-0006"),
    (pudl["BA"] == "MISO-8910") & (pudl["state"] == "AR"),
    (pudl["BA"] == "MISO-8910") & (pudl["state"].isin(["LA", "TX"])),
    (pudl["BA"] == "MISO-8910") & (pudl["state"] == "MS"),
]

choices = ["01", "02", "07", "03", "05", "04", "06", "08", "09", "10"]

pudl["LRZ"] = np.select(conditions, choices, default=None)
pudl = pudl[pudl["LRZ"].notna()]

### Solar Profiles

In [ ]:
solar = pudl[pudl["technology_description"] == "Solar Photovoltaic"]

In [2]:
from pathlib import Path
from plotly.subplots import make_subplots

WEIGHTED_DIR = Path("Mapping/capacity_weighted")
ENSEMBLES    = [f"{i:03d}" for i in range(1, 20)]  # 001–019

lrzs = sorted(lrz for p in (WEIGHTED_DIR / ENSEMBLES[0]).glob("solar_*.csv")
              for lrz in [p.stem.split("_", 1)[1]])
lrz_titles = {"01":"ND & MN", "02":"WI", "03":"IA", "04":"IL", "05":"MO", "06": "IN", "07": "MI", "08" : "AK", "09": "LA", "10": "MS"}
lrz_titles_long = {"01":"North Dakota & Minnesota", "02":"Wisconsin", "03":"Iowa", "04":"Illinois", "05":"Missouri", "06": "Indiana", "07": "Michigan", "08" : "Arkansas", "09": "Louisiana", "10": "Mississippi"}

# Annual mean CF per (LRZ, ensemble, year) from pre-computed weighted profiles
records = []
for ens in ENSEMBLES:
    ens_dir = WEIGHTED_DIR / ens
    for lrz in lrzs:
        df = pd.read_csv(ens_dir / f"solar_{lrz}.csv", usecols=["time", "cf"])
        df["year"] = pd.to_datetime(df["time"]).dt.year
        for year, annual_cf in df.groupby("year")["cf"].mean().items():
            records.append({"LRZ": lrz, "ensemble": ens, "year": year, "wtd_cf": annual_cf})

lrz_cf = pd.DataFrame(records)

# Histogram per LRZ
n_cols = 5
n_rows = -(-len(lrzs) // n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f"LRZ {lrz}" for lrz in lrzs],
)

for i, lrz in enumerate(lrzs):
    row, col = i // n_cols + 1, i % n_cols + 1
    vals = lrz_cf[lrz_cf["LRZ"] == lrz]["wtd_cf"]
    fig.add_trace(
        go.Histogram(x=vals, name=f"LRZ {lrz}", showlegend=False, marker_color="steelblue"),
        row=row, col=col,
    )

fig.update_xaxes(title_text="Annual CF")
fig.update_layout(
    height=320 * n_rows,
    title_text="Capacity-Weighted Mean Annual Solar CF by LRZ (Ensembles 001–019, 2015–2045)",
)
fig.show()


### 3-Day Rolling Solar Production by LRZ

In [ ]:
from pathlib import Path
from plotly.subplots import make_subplots

WEIGHTED_DIR = Path("Mapping/capacity_weighted")
ENSEMBLES    = [f"{i:03d}" for i in range(1, 20)]  # 001–019

lrzs = sorted(lrz for p in (WEIGHTED_DIR / ENSEMBLES[0]).glob("solar_*.csv")
              for lrz in [p.stem.split("_", 1)[1]])

rolling_dfs = []

for ens in ENSEMBLES:
    ens_dir = WEIGHTED_DIR / ens

    cf_cols    = {}
    time_index = None
    for lrz in lrzs:
        df = pd.read_csv(ens_dir / f"solar_{lrz}.csv", usecols=["time", "cf"])
        if time_index is None:
            time_index = pd.to_datetime(df["time"])
        cf_cols[lrz] = df["cf"].values

    cf_df   = pd.DataFrame(cf_cols, index=time_index)
    rolling = cf_df.rolling(12).mean()  # 3-day mean CF

    long = rolling.dropna().stack().rename("value").reset_index()
    long.columns = ["time", "LRZ", "value"]
    long["ensemble"] = ens
    rolling_dfs.append(long[["LRZ", "ensemble", "value"]])

all_rolling_solar = pd.concat(rolling_dfs, ignore_index=True)

# Histogram per LRZ
n_cols = 5
n_rows = -(-len(lrzs) // n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f"{lrz_titles_long[lrz]}" for lrz in lrzs],
)

for i, lrz in enumerate(lrzs):
    row, col = i // n_cols + 1, i % n_cols + 1
    vals = all_rolling_solar.loc[all_rolling_solar["LRZ"] == lrz, "value"]
    fig.add_trace(
        go.Histogram(x=vals, name=f"{lrz_titles_long[lrz]}", showlegend=False, marker_color="steelblue"),
        row=row, col=col,
    )

fig.update_xaxes(title_text="3-day mean CF")
fig.update_layout(
    height=320 * n_rows,
    title_text="3-Day Rolling Mean Solar CF by LRZ (Ensembles 001–019, 2015–2045)",
    template="plotly_white",
)
fig.show()


In [ ]:
# Map: x-axis label -> quantile value
return_periods = {
    100: 1/(100*365),
    50:  1/(50*365),
    20:  1/(20*365),
    10:  1/(10*365),
    2:   1/(2*365),
}

pct_df = pd.DataFrame(
    {rp: all_rolling_solar.groupby("LRZ")["value"].quantile(q) for rp, q in return_periods.items()}
).reset_index()

lrz_list = sorted(pct_df["LRZ"].unique())
n_cols = 2
n_rows = (len(lrz_list) + n_cols - 1) // n_cols


fig2 = make_subplots(
    rows=n_rows,
    cols=n_cols,
    shared_yaxes=False,
    subplot_titles=[lrz_titles[lrz] for lrz in lrz_list],
    vertical_spacing=0.08,
    horizontal_spacing=0.08,
)

x_vals = sorted(return_periods.keys())  # [2, 10, 20, 50, 100]

for i, lrz in enumerate(lrz_list):
    row = i // n_cols + 1
    col = i % n_cols + 1

    lrz_row = pct_df[pct_df["LRZ"] == lrz]
    y_vals = [lrz_row[rp].values[0] for rp in x_vals]

    fig2.add_trace(
        go.Scatter(
            x=x_vals,
            y=y_vals,
            mode="lines+markers",
            line=dict(width=2),
            marker=dict(size=7),
            showlegend=False,
        ),
        row=row,
        col=col,
    )

fig2.update_xaxes(
    tickvals=x_vals,
    ticktext=[str(rp) for rp in x_vals],
    title_text="1-in-X",
)
fig2.update_yaxes(title_text="CF (%)")

fig2.update_layout(
    height=200 * n_rows,
    title_text="3-Day Solar Production, all members",
    template="plotly_white",
)

fig2.show()

In [ ]:
from pathlib import Path
from plotly.subplots import make_subplots

WEIGHTED_DIR = Path("Mapping/capacity_weighted")
ENSEMBLES    = [f"{i:03d}" for i in range(1, 20)]  # 001–019

lrzs = sorted(lrz for p in (WEIGHTED_DIR / ENSEMBLES[0]).glob("wind_*.csv")
              for lrz in [p.stem.split("_", 1)[1]])

rolling_dfs = []

for ens in ENSEMBLES:
    ens_dir = WEIGHTED_DIR / ens

    cf_cols    = {}
    time_index = None
    for lrz in lrzs:
        df = pd.read_csv(ens_dir / f"wind_{lrz}.csv", usecols=["time", "cf"])
        if time_index is None:
            time_index = pd.to_datetime(df["time"])
        cf_cols[lrz] = df["cf"].values

    cf_df   = pd.DataFrame(cf_cols, index=time_index)
    rolling = cf_df.rolling(12).mean()  # 3-day mean CF

    long = rolling.dropna().stack().rename("value").reset_index()
    long.columns = ["time", "LRZ", "value"]
    long["ensemble"] = ens
    rolling_dfs.append(long[["LRZ", "ensemble", "value"]])

all_rolling_wind = pd.concat(rolling_dfs, ignore_index=True)

# Histogram per LRZ
n_cols = 5
n_rows = -(-len(lrzs) // n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f"{lrz_titles_long[lrz]}" for lrz in lrzs],
)

for i, lrz in enumerate(lrzs):
    row, col = i // n_cols + 1, i % n_cols + 1
    vals = all_rolling_wind.loc[all_rolling_wind["LRZ"] == lrz, "value"]
    fig.add_trace(
        go.Histogram(x=vals, name=f"LRZ {lrz}", showlegend=False, marker_color="steelblue"),
        row=row, col=col,
    )

fig.update_xaxes(title_text="3-day mean CF")
fig.update_layout(
    height=320 * n_rows,
    title_text="3-Day Rolling Mean Wind CF by LRZ (Ensembles 001–019, 2015–2045)",
    template="plotly_white",
)
fig.show()


In [ ]:
# Map: x-axis label -> quantile value
return_periods = {
    100: 1/(100*365),
    50:  1/(50*365),
    20:  1/(20*365),
    10:  1/(10*365),
    2:   1/(2*365),
}

pct_df = pd.DataFrame(
    {rp: all_rolling_wind.groupby("LRZ")["value"].quantile(q) for rp, q in return_periods.items()}
).reset_index()

lrz_list = sorted(pct_df["LRZ"].unique())
n_cols = 2
n_rows = (len(lrz_list) + n_cols - 1) // n_cols

fig2 = make_subplots(
    rows=n_rows,
    cols=n_cols,
    shared_yaxes=False,
    subplot_titles=[lrz_titles[lrz] for lrz in lrz_list],
    vertical_spacing=0.08,
    horizontal_spacing=0.08,
)

x_vals = sorted(return_periods.keys())  # [2, 10, 20, 50, 100]

for i, lrz in enumerate(lrz_list):
    row = i // n_cols + 1
    col = i % n_cols + 1

    lrz_row = pct_df[pct_df["LRZ"] == lrz]
    y_vals = [lrz_row[rp].values[0] for rp in x_vals]

    fig2.add_trace(
        go.Scatter(
            x=x_vals,
            y=y_vals,
            mode="lines+markers",
            line=dict(width=2),
            marker=dict(size=7),
            showlegend=False,
        ),
        row=row,
        col=col,
    )

fig2.update_xaxes(
    tickvals=x_vals,
    ticktext=[str(rp) for rp in x_vals],
    title_text="1-in-X",
)
fig2.update_yaxes(title_text="CF (%)")

fig2.update_layout(
    height=200 * n_rows,
    title_text="3-Day Wind Production, all members",
    template="plotly_white",
)

fig2.show()

In [ ]:
# ── Capacity totals per LRZ (matches what write_capacity_weighted used) ────────
solar_cap = pudl.dropna(subset=["solar_profile_name"]).groupby("LRZ")["capacity_mw"].sum()
wind_cap  = pudl.dropna(subset=["wind_profile_name"]).groupby("LRZ")["capacity_mw"].sum()
all_lrzs  = sorted(set(solar_cap.index) | set(wind_cap.index))

# ── Build combined rolling: weight solar + wind by MW at time-series level ─────
combined_dfs = []
for ens in ENSEMBLES:
    ens_dir = WEIGHTED_DIR / ens

    cf_cols    = {}
    time_index = None

    for lrz in all_lrzs:
        s_cap     = solar_cap.get(lrz, 0.0)
        w_cap     = wind_cap.get(lrz, 0.0)
        total_cap = s_cap + w_cap
        if total_cap == 0:
            continue

        s_path, w_path = ens_dir / f"solar_{lrz}.csv", ens_dir / f"wind_{lrz}.csv"

        solar_cf = 0.0
        if s_cap > 0 and s_path.exists():
            df_s = pd.read_csv(s_path, usecols=["time", "cf"])
            if time_index is None:
                time_index = pd.to_datetime(df_s["time"])
            solar_cf = df_s["cf"].values

        wind_cf = 0.0
        if w_cap > 0 and w_path.exists():
            df_w = pd.read_csv(w_path, usecols=["time", "cf"])
            if time_index is None:
                time_index = pd.to_datetime(df_w["time"])
            wind_cf = df_w["cf"].values

        cf_cols[lrz] = (s_cap * solar_cf + w_cap * wind_cf) / total_cap

    cf_df   = pd.DataFrame(cf_cols, index=time_index)
    rolling = cf_df.rolling(12).mean()  # 12 × 6-h steps = 3-day rolling mean

    long = rolling.dropna().stack().rename("value").reset_index()
    long.columns = ["time", "LRZ", "value"]
    long["ensemble"] = ens
    combined_dfs.append(long[["LRZ", "ensemble", "value"]])

all_rolling_combined = pd.concat(combined_dfs, ignore_index=True)

# ── Histogram per LRZ ─────────────────────────────────────────────────────────
lrzs_c = sorted(all_rolling_combined["LRZ"].unique())
n_cols  = 5
n_rows  = -(-len(lrzs_c) // n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[lrz_titles_long[lrz] for lrz in lrzs_c],
)

for i, lrz in enumerate(lrzs_c):
    row, col = i // n_cols + 1, i % n_cols + 1
    vals = all_rolling_combined.loc[all_rolling_combined["LRZ"] == lrz, "value"]
    fig.add_trace(
        go.Histogram(x=vals, name=lrz_titles_long[lrz], showlegend=False, marker_color="steelblue"),
        row=row, col=col,
    )

fig.update_xaxes(title_text="3-day mean CF")
fig.update_layout(
    height=320 * n_rows,
    title_text="3-Day Rolling Combined Wind + Solar CF by LRZ (Ensembles 001–019, 2015–2045)",
    template="plotly_white",
)
fig.show()


### Month-Hour Capacity Factor Plots

In [ ]:
SOLAR_DIR = Path("Solar Profiles Clean")
WIND_DIR  = Path("Wind Profiles Clean")
ENSEMBLES = [f"{i:03d}" for i in range(1, 20)]  # 001–019

MONTH_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]


solar_df = pd.DataFrame(columns=lrzs)
for ens in ENSEMBLES:
    for lrz in lrzs: 
        df = pd.read_csv(f"Mapping/capacity_weighted/{ens}/solar_{lrz}.csv")
        
        if solar_df[lrz].isna().all():
            time_index = (pd.to_datetime(df["time"])
                                .dt.tz_localize("UTC")
                                .dt.tz_convert("US/Central"))
            solar_df[lrz] = 1/len(ENSEMBLES)*df["cf"].values
            solar_df.index = time_index
        else:
            solar_df[lrz] += 1/len(ENSEMBLES)*df["cf"].values


solar_mh = solar_df.groupby([solar_df.index.month, solar_df.index.hour]).mean()
idx = list(range(0,8)) + [9, 11, 13, 15] + list(range(16, 44)) + [44, 46, 48, 50] + list(range(52,56))
solar_mh = solar_mh.iloc[idx]
 
hour_labels = [f"{h:02d}:00" for h in solar_mh.index.get_level_values(1)]

# Color scheme: LRZ 01–07 red scale, 08–10 blue scale (darker = higher LRZ)
blue_lrzs  = ["01", "02", "03", "04", "05", "06", "07"]
red_lrzs = ["08", "09", "10"]

blue_colors  = plotly.colors.sample_colorscale("Blues",  [0.25 + 0.6 * i / (len(blue_lrzs)  - 1) for i in range(len(blue_lrzs))])
red_colors = plotly.colors.sample_colorscale("Reds", [0.35 + 0.5 * i / (len(red_lrzs) - 1) for i in range(len(red_lrzs))])

lrz_colors = {lrz: red_colors[i]  for i, lrz in enumerate(red_lrzs)}
lrz_colors.update({lrz: blue_colors[i] for i, lrz in enumerate(blue_lrzs)})


fig = go.Figure()

for lrz in lrzs:
    fig.add_trace(go.Scatter(
        x=[[month for month in MONTH_LABELS for _ in range(4)], hour_labels],
        y=solar_mh[lrz].values,
        name=lrz,
        line=dict(color=lrz_colors[lrz]),
    ))

fig.update_xaxes(title_text="Hour (Central)")
fig.update_yaxes(title_text="Mean CF")
fig.update_layout(
    height=450,
    title_text="Month-Hour Capacity Factor: Solar (all members)",
    template="plotly_white",
    legend=dict(title="LRZ"),
    font_size=18
)
fig.show()

In [ ]:
WIND_DIR  = Path("Wind Profiles Clean")
ENSEMBLES = [f"{i:03d}" for i in range(1, 20)]  # 001–019

MONTH_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# Color scheme: LRZ 01–07 red scale, 08–10 blue scale (darker = higher LRZ)
blue_lrzs  = ["01", "02", "03", "04", "05", "06", "07"]
red_lrzs = ["08", "09", "10"]

blue_colors  = plotly.colors.sample_colorscale("Blues",  [0.25 + 0.6 * i / (len(blue_lrzs)  - 1) for i in range(len(blue_lrzs))])
red_colors = plotly.colors.sample_colorscale("Reds", [0.35 + 0.5 * i / (len(red_lrzs) - 1) for i in range(len(red_lrzs))])

lrz_colors = {lrz: red_colors[i]  for i, lrz in enumerate(red_lrzs)}
lrz_colors.update({lrz: blue_colors[i] for i, lrz in enumerate(blue_lrzs)})

wind_df = pd.DataFrame(columns=lrzs)
for ens in ENSEMBLES:
    for lrz in lrzs: 
        if lrz == "09": continue
        df = pd.read_csv(f"Mapping/capacity_weighted/{ens}/wind_{lrz}.csv")
        
        if wind_df[lrz].isna().all():
            time_index = (pd.to_datetime(df["time"])
                                .dt.tz_localize("UTC")
                                .dt.tz_convert("US/Central"))
            wind_df[lrz] = 1/len(ENSEMBLES)*df["cf"].values
            wind_df.index = time_index
        else:
            wind_df[lrz] += 1/len(ENSEMBLES)*df["cf"].values


wind_mh = wind_df.groupby([wind_df.index.month, wind_df.index.hour]).mean()
idx = list(range(0,8)) + [9, 11, 13, 15] + list(range(16, 44)) + [44, 46, 48, 50] + list(range(52,56))
wind_mh = wind_mh.iloc[idx]
 
hour_labels = [f"{h:02d}:00" for h in wind_mh.index.get_level_values(1)]

fig = go.Figure()

for lrz in lrzs:
    fig.add_trace(go.Scatter(
        x=[[month for month in MONTH_LABELS for _ in range(4)], hour_labels],
        y=wind_mh[lrz].values,
        name=lrz,
        line=dict(color=lrz_colors[lrz]),
    ))

fig.update_xaxes(title_text="Hour (Central)")
fig.update_yaxes(title_text="Mean CF")
fig.update_layout(
    height=450,
    title_text="Month-Hour Capacity Factor: Wind (all members)",
    template="plotly_white",
    legend=dict(title="LRZ"),
    font_size=18,
)
fig.show()


In [ ]:
import plotly.colors

WIND_DIR  = Path("Wind Profiles Clean")
ENSEMBLES = [f"{i:03d}" for i in range(1, 20)]  # 001–019

MONTH_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# Color scheme: LRZ 01–07 red scale, 08–10 blue scale (darker = higher LRZ)
blue_lrzs  = ["01", "02", "03", "04", "05", "06", "07"]
red_lrzs = ["08", "09", "10"]

blue_colors  = plotly.colors.sample_colorscale("Blues",  [0.25 + 0.6 * i / (len(blue_lrzs)  - 1) for i in range(len(blue_lrzs))])
red_colors = plotly.colors.sample_colorscale("Reds", [0.35 + 0.5 * i / (len(red_lrzs) - 1) for i in range(len(red_lrzs))])

lrz_colors = {lrz: red_colors[i]  for i, lrz in enumerate(red_lrzs)}
lrz_colors.update({lrz: blue_colors[i] for i, lrz in enumerate(blue_lrzs)})

wind_df = pd.DataFrame(columns=lrzs)
for ens in ENSEMBLES:
    for lrz in lrzs: 
        if lrz == "09": continue
        df = pd.read_csv(f"Mapping/capacity_weighted/{ens}/wind_{lrz}.csv")
        
        if wind_df[lrz].isna().all():
            time_index = (pd.to_datetime(df["time"])
                                .dt.tz_localize("UTC")
                                .dt.tz_convert("US/Central"))
            wind_df[lrz] = 1/len(ENSEMBLES)*df["cf"].values
            wind_df.index = time_index
        else:
            wind_df[lrz] += 1/len(ENSEMBLES)*df["cf"].values


wind_mh = wind_df.groupby([wind_df.index.month]).mean()

fig = go.Figure()

for lrz in lrzs:
    fig.add_trace(go.Scatter(
        x=MONTH_LABELS,
        y=wind_mh[lrz].values,
        name=lrz,
        line=dict(color=lrz_colors[lrz]),
    ))

fig.update_xaxes(title_text="Month")
fig.update_yaxes(title_text="Mean CF")
fig.update_layout(
    height=450,
    title_text="Monthly Capacity Factor: Wind (all members)",
    template="plotly_white",
    legend=dict(title="LRZ"),
    font_size=18,
)
fig.show()


## Create Capacity Weighted Profiles

In [ ]:
from pathlib import Path

SOLAR_DIR = Path("Solar Profiles Clean")
WIND_DIR  = Path("Wind Profiles Clean")
ENSEMBLES = [f"{i:03d}" for i in range(1, 20)]  # 001–019
OUT_DIR   = Path("Mapping/capacity_weighted")

# Capacity tables with LRZ column (solar_cap_lrz / wind_cap_lrz from cell above)
solar_cap_lrz = (
    solar.dropna(subset=["solar_profile_name", "LRZ", "capacity_mw"])
    .groupby(["LRZ", "solar_profile_name"])["capacity_mw"].sum()
    .reset_index().rename(columns={"solar_profile_name": "profile"})
)
wind_cap_lrz = (
    wind.dropna(subset=["wind_profile_name", "LRZ", "capacity_mw"])
    .groupby(["LRZ", "wind_profile_name"])["capacity_mw"].sum()
    .reset_index().rename(columns={"wind_profile_name": "profile"})
)


def write_capacity_weighted(profiles_dir, cap_df, resource, ensembles, out_dir):
    lrz_map = cap_df.set_index("profile")["LRZ"]
    lrz_cap = cap_df.groupby("LRZ")["capacity_mw"].sum()

    for ens in ensembles:
        ens_dir = profiles_dir / ens
        ens_out = out_dir / ens
        ens_out.mkdir(parents=True, exist_ok=True)

        time_col  = None
        prod_cols = {}

        for _, row in cap_df.iterrows():
            df = pd.read_csv(ens_dir / f"{row['profile']}.csv", usecols=["time", "cf"])
            if time_col is None:
                time_col = df["time"]
            prod_cols[row["profile"]] = df["cf"].values * row["capacity_mw"]

        prod_df  = pd.DataFrame(prod_cols)
        lrz_prod = prod_df.T.groupby(lrz_map).sum().T   # timestep × LRZ (MW)
        cf_df    = lrz_prod.div(lrz_cap, axis=1)         # capacity-weighted CF

        for lrz in cf_df.columns:
            out = pd.DataFrame({"time": time_col.values, "cf": cf_df[lrz].values})
            out.to_csv(ens_out / f"{resource}_{lrz}.csv", index=False)

        print(f"  [{ens}] wrote {len(cf_df.columns)} {resource} LRZ profiles")


write_capacity_weighted(SOLAR_DIR, solar_cap_lrz, "solar", ENSEMBLES, OUT_DIR)
write_capacity_weighted(WIND_DIR,  wind_cap_lrz,  "wind",  ENSEMBLES, OUT_DIR)
print("Done.")


In [7]:
from pathlib import Path

WIND_DIR = Path("/glade/work/rbhandarkar/Inputs/Wind Profiles/ERA5/")
SOLAR_DIR = Path("/glade/work/rbhandarkar/Inputs/Solar Profiles/ERA5/")
OUT_DIR  = Path("Mapping/capacity_weighted/ERA5")
YEARS    = range(2015, 2026)

wind = pd.read_csv("/glade/u/home/rbhandarkar/Inputs/Generator Data/Mapping/wind_era5_map.csv")
solar = pd.read_csv("/glade/u/home/rbhandarkar/Inputs/Generator Data/Mapping/solar_era5_map.csv")

wind_cap_lrz = (
    wind.dropna(subset=["profile_name", "LRZ", "capacity_mw"])
    .groupby(["LRZ", "profile_name"])["capacity_mw"].sum()
    .reset_index().rename(columns={"profile_name": "profile"})
)
solar_cap_lrz = (
    solar.dropna(subset=["profile_name", "LRZ", "capacity_mw"])
    .groupby(["LRZ", "profile_name"])["capacity_mw"].sum()
    .reset_index().rename(columns={"profile_name": "profile"})
)

def write_capacity_weighted_era5(profiles_dir, cap_df, resource, years, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    lrz_map = cap_df.set_index("profile")["LRZ"]
    lrz_cap = cap_df.groupby("LRZ")["capacity_mw"].sum()

    time_col  = None
    prod_cols = {}

    for _, row in cap_df.iterrows():
        yearly = []
        for year in years:
            fpath = profiles_dir / f"{row['profile']}_{year}.csv"
            yearly.append(pd.read_csv(fpath, usecols=["time", "cf"]))
        df = pd.concat(yearly, ignore_index=True)
        if time_col is None:
            time_col = df["time"]
        prod_cols[row["profile"]] = df["cf"].values * row["capacity_mw"]

    prod_df  = pd.DataFrame(prod_cols)
    lrz_prod = prod_df.T.groupby(lrz_map).sum().T  # timestep × LRZ (MW)
    cf_df    = lrz_prod.div(lrz_cap, axis=1)        # capacity-weighted CF

    for lrz in cf_df.columns:
        out = pd.DataFrame({"time": time_col.values, "cf": cf_df[lrz].values})
        out.to_csv(out_dir / f"era5_{resource}_{lrz}.csv", index=False)

    print(f"Wrote {len(cf_df.columns)} {resource} LRZ profiles ({years[0]}–{years[-1]}) → {out_dir}")

# write_capacity_weighted_era5(WIND_DIR, wind_cap_lrz, "wind", YEARS, OUT_DIR)
write_capacity_weighted_era5(SOLAR_DIR, solar_cap_lrz, "solar", YEARS, OUT_DIR)
print("Done.")


Wrote 10 solar LRZ profiles (2015–2025) → Mapping/capacity_weighted/ERA5
Done.


In [12]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors

WEIGHTED_DIR = Path("Mapping/capacity_weighted")
ENSEMBLES    = [f"{i:03d}" for i in range(1, 20)]
LRZS         = ["01", "02", "03", "04", "05", "06", "07", "08", "10"]
MONTH_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# ── Load ERA5 monthly means ───────────────────────────────────────────────────
era5_monthly = {}
for lrz in LRZS:
    df = pd.read_csv(WEIGHTED_DIR / "ERA5" / f"era5_wind_{int(lrz)}.csv")
    df.index = (pd.to_datetime(df["time"])
                .dt.tz_localize("UTC")
                .dt.tz_convert("US/Central"))
    era5_monthly[lrz] = df["cf"].groupby(df.index.month).mean().values  # (12,)

# ── Load CESM: per-ensemble monthly means → ensemble spread ──────────────────
cesm_monthly = {lrz: [] for lrz in LRZS}  # lrz → list of (12,) arrays, one per ens

for ens in ENSEMBLES:
    for lrz in LRZS:
        f = WEIGHTED_DIR / ens / f"wind_{lrz}.csv"
        if not f.exists():
            continue
        df = pd.read_csv(f)
        df.index = (pd.to_datetime(df["time"])
                    .dt.tz_localize("UTC")
                    .dt.tz_convert("US/Central"))
        df = df[df.index.year.isin(range(2015, 2026))]   # ← filter to 2015–2025
        cesm_monthly[lrz].append(df["cf"].groupby(df.index.month).mean().values)

# ── Plot ──────────────────────────────────────────────────────────────────────
ncols = 3
nrows = -(-len(LRZS) // ncols)

lrz_titles_long = {
    "01": "LRZ 01 — ND & MN", "02": "LRZ 02 — WI",  "03": "LRZ 03 — IA",
    "04": "LRZ 04 — IL",      "05": "LRZ 05 — MO",  "06": "LRZ 06 — IN",
    "07": "LRZ 07 — MI",      "08": "LRZ 08 — AR",  "10": "LRZ 10 — MS",
}

fig = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[lrz_titles_long[lrz] for lrz in LRZS],
    shared_yaxes=False,
    vertical_spacing=0.1,
)

for i, lrz in enumerate(LRZS):
    row, col = i // ncols + 1, i % ncols + 1
    show_legend = (i == 0)

    ens_matrix = np.array(cesm_monthly[lrz])        # (n_ens, 12)
    cesm_mean  = ens_matrix.mean(axis=0)
    cesm_p10   = np.percentile(ens_matrix, 10, axis=0)
    cesm_p90   = np.percentile(ens_matrix, 90, axis=0)

    # CESM shaded band
    fig.add_trace(go.Scatter(
        x=MONTH_LABELS + MONTH_LABELS[::-1],
        y=np.concatenate([cesm_p90, cesm_p10[::-1]]),
        fill="toself", fillcolor="rgba(70,130,180,0.2)",
        line=dict(width=0), hoverinfo="skip",
        legendgroup="CESM", showlegend=False,
    ), row=row, col=col)

    # CESM mean line
    fig.add_trace(go.Scatter(
        x=MONTH_LABELS, y=cesm_mean,
        name="CESM HR (mean ± p10/p90)",
        line=dict(color="steelblue", width=2),
        legendgroup="CESM", showlegend=show_legend,
    ), row=row, col=col)

    # ERA5 line
    fig.add_trace(go.Scatter(
        x=MONTH_LABELS, y=era5_monthly[lrz],
        name="ERA5",
        line=dict(color="tomato", width=2),
        legendgroup="ERA5", showlegend=show_legend,
    ), row=row, col=col)

fig.update_yaxes(title_text="Mean CF", range=[0, None])
fig.update_xaxes(title_text="Month")
fig.update_layout(
    height=320 * nrows,
    width=380 * ncols,
    title_text="Monthly Wind CF by LRZ — CESM HR vs ERA5 (2015–2025)",
    template="plotly_white",
    font_size=13,
)
fig.show()


In [9]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors

WEIGHTED_DIR = Path("Mapping/capacity_weighted")
ENSEMBLES    = [f"{i:03d}" for i in range(1, 20)]
LRZS         = ["01", "02", "03", "04", "05", "06", "07", "08", "10"]
MONTH_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# ── Load ERA5 monthly means ───────────────────────────────────────────────────
era5_monthly = {}
for lrz in LRZS:
    df = pd.read_csv(WEIGHTED_DIR / "ERA5" / f"era5_solar_{int(lrz)}.csv")
    df.index = (pd.to_datetime(df["time"])
                .dt.tz_localize("UTC")
                .dt.tz_convert("US/Central"))
    era5_monthly[lrz] = df["cf"].groupby(df.index.month).mean().values  # (12,)

# ── Load CESM: per-ensemble monthly means → ensemble spread ──────────────────
cesm_monthly = {lrz: [] for lrz in LRZS}  # lrz → list of (12,) arrays, one per ens

for ens in ENSEMBLES:
    for lrz in LRZS:
        f = WEIGHTED_DIR / ens / f"solar_{lrz}.csv"
        if not f.exists():
            continue
        df = pd.read_csv(f)
        df.index = (pd.to_datetime(df["time"])
                    .dt.tz_localize("UTC")
                    .dt.tz_convert("US/Central"))
        df = df[df.index.year.isin(range(2015, 2026))]   # ← filter to 2015–2025
        cesm_monthly[lrz].append(df["cf"].groupby(df.index.month).mean().values)

# ── Plot ──────────────────────────────────────────────────────────────────────
ncols = 3
nrows = -(-len(LRZS) // ncols)

lrz_titles_long = {
    "01": "LRZ 01 — ND & MN", "02": "LRZ 02 — WI",  "03": "LRZ 03 — IA",
    "04": "LRZ 04 — IL",      "05": "LRZ 05 — MO",  "06": "LRZ 06 — IN",
    "07": "LRZ 07 — MI",      "08": "LRZ 08 — AR",  "10": "LRZ 10 — MS",
}

fig = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[lrz_titles_long[lrz] for lrz in LRZS],
    shared_yaxes=False,
    vertical_spacing=0.1,
)

for i, lrz in enumerate(LRZS):
    row, col = i // ncols + 1, i % ncols + 1
    show_legend = (i == 0)

    ens_matrix = np.array(cesm_monthly[lrz])        # (n_ens, 12)
    cesm_mean  = ens_matrix.mean(axis=0)
    cesm_p10   = np.percentile(ens_matrix, 10, axis=0)
    cesm_p90   = np.percentile(ens_matrix, 90, axis=0)

    # CESM shaded band
    fig.add_trace(go.Scatter(
        x=MONTH_LABELS + MONTH_LABELS[::-1],
        y=np.concatenate([cesm_p90, cesm_p10[::-1]]),
        fill="toself", fillcolor="rgba(70,130,180,0.2)",
        line=dict(width=0), hoverinfo="skip",
        legendgroup="CESM", showlegend=False,
    ), row=row, col=col)

    # CESM mean line
    fig.add_trace(go.Scatter(
        x=MONTH_LABELS, y=cesm_mean,
        name="CESM HR (mean ± p10/p90)",
        line=dict(color="steelblue", width=2),
        legendgroup="CESM", showlegend=show_legend,
    ), row=row, col=col)

    # ERA5 line
    fig.add_trace(go.Scatter(
        x=MONTH_LABELS, y=era5_monthly[lrz],
        name="ERA5",
        line=dict(color="tomato", width=2),
        legendgroup="ERA5", showlegend=show_legend,
    ), row=row, col=col)

fig.update_yaxes(title_text="Mean CF", range=[0, None])
fig.update_xaxes(title_text="Month")
fig.update_layout(
    height=350 * nrows,
    width=400 * ncols,
    title_text="Monthly Solar CF by LRZ — CESM HR vs ERA5 (2015–2025)",
    template="plotly_white",
    font_size=13,
)
fig.show()


## Compare to ERA5